# 02 – Data Cleaning
Demonstrates the cleaning pipeline: dedup, missing values, outliers, coordinate validation.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.clean_data import (
    drop_high_missing, impute_missing, remove_outliers,
    validate_coordinates, validate_severity, clean_strings
)
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Load interim parquet (already created by make_dataset.py)
df = pd.read_parquet('../data/interim/accidents_cleaned.parquet')
print(f'Loaded: {df.shape}')

In [ ]:
# Before cleaning: missing value summary
missing_before = df.isnull().mean().sort_values(ascending=False)
print('Top missing columns before cleaning:')
print(missing_before[missing_before > 0].head(10).round(3))

In [ ]:
# Step 1: Deduplication
n_before = len(df)
df = df.drop_duplicates(subset=['ID'])
print(f'Removed {n_before - len(df):,} duplicates. Remaining: {len(df):,}')

In [ ]:
# Step 2: Drop high-missing columns
df = drop_high_missing(df, threshold=0.5)
print(f'Shape after dropping high-missing cols: {df.shape}')

In [ ]:
# Step 3: Coordinate validation
n_before = len(df)
df = validate_coordinates(df)
print(f'Removed {n_before - len(df):,} rows with invalid coordinates')

In [ ]:
# Step 4: Severity validation
df = validate_severity(df)
print(f'Shape after severity validation: {df.shape}')

In [ ]:
# Step 5: Outlier analysis before capping
num_cols = ['Distance(mi)', 'Temperature(F)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
fig, axes = plt.subplots(1, len(num_cols), figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    if col in df.columns:
        df[col].dropna().clip(df[col].quantile(0.01), df[col].quantile(0.99)).hist(ax=ax, bins=40, color='steelblue', edgecolor='white')
        ax.set_title(col, fontsize=9)
plt.suptitle('Numeric Distributions (1st-99th percentile)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Outlier capping
df = remove_outliers(df, factor=3.0)
print('Outliers capped with IQR factor=3.0')

In [ ]:
# Step 7: Imputation
df = impute_missing(df)
print(f'Missing values after imputation: {df.isnull().sum().sum()}')

In [ ]:
# Step 8: String cleaning
df = clean_strings(df)
print(f'Final cleaned shape: {df.shape}')
print(f'Severity distribution:\n{df["Severity"].value_counts()}')